In [6]:
!pip install kagglehub
import kagglehub, os, pandas as pd

path = kagglehub.dataset_download("teejmahal20/airline-passenger-satisfaction")
print("Dataset folder:", path)
print("Files:", os.listdir(path))

  Using cached kagglehub-1.0.0-py3-none-any.whl.metadata (40 kB)
Using cached kagglehub-1.0.0-py3-none-any.whl (70 kB)
Dataset folder: /home/sagemaker-user/.cache/kagglehub/datasets/teejmahal20/airline-passenger-satisfaction/versions/1
Files: ['test.csv', 'train.csv']


In [7]:
files = os.listdir(path)

if "train.csv" in files and "test.csv" in files:
    train_df = pd.read_csv(os.path.join(path, "train.csv"))
    test_df  = pd.read_csv(os.path.join(path, "test.csv"))
    df = pd.concat([train_df, test_df], ignore_index=True)
else:
    csvs = [f for f in files if f.endswith(".csv")]
    df = pd.read_csv(os.path.join(path, csvs[0]))

df.head()

,Unnamed: 0,id,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,...,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction
0,0,70172,Male,Loyal Customer,13,Personal Travel,Eco Plus,460,3,4,...,5,4,3,4,4,5,5,25,18.0,neutral or dissatisfied
1,1,5047,Male,disloyal Customer,25,Business travel,Business,235,3,2,...,1,1,5,3,1,4,1,1,6.0,neutral or dissatisfied
2,2,110028,Female,Loyal Customer,26,Business travel,Business,1142,2,2,...,5,4,3,4,4,4,5,0,0.0,satisfied
3,3,24026,Female,Loyal Customer,25,Business travel,Business,562,2,5,...,2,2,5,3,1,4,2,11,9.0,neutral or dissatisfied
4,4,119299,Male,Loyal Customer,61,Business travel,Business,214,3,3,...,3,3,4,4,3,3,3,0,0.0,satisfied


In [8]:
df = df.drop(columns=["Unnamed: 0", "id"], errors="ignore")
df["satisfaction"] = df["satisfaction"].map({"satisfied": 1, "neutral or dissatisfied": 0})
df["satisfaction"].value_counts()

satisfaction
0    73452
1    56428
Name: count, dtype: int64

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import joblib, os

X = df.drop(columns=["satisfaction"])
y = df["satisfaction"]

cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
num_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                          ("scaler", StandardScaler())]), num_cols),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_cols)
    ]
)

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=42)
X_valid, X_test, y_valid, y_test = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42)

clf = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=200))
])

clf.fit(X_train, y_train)

def eval_set(name, Xs, ys):
    proba = clf.predict_proba(Xs)[:,1]
    pred = (proba >= 0.5).astype(int)
    return name, accuracy_score(ys, pred), f1_score(ys, pred), roc_auc_score(ys, proba)

print("VALID (acc, f1, auc):", eval_set("valid", X_valid, y_valid))
print("TEST  (acc, f1, auc):", eval_set("test", X_test, y_test))

VALID (acc, f1, auc): ('valid', 0.8734729493891797, 0.8514612835191323, 0.926597096570966)
TEST  (acc, f1, auc): ('test', 0.8739862437121445, 0.8517422549670873, 0.9276691200616554)


In [10]:
os.makedirs("../model", exist_ok=True)
joblib.dump(clf, "../model/model.pkl")
print("Saved to ../model/model.pkl")

Saved to ../model/model.pkl


In [11]:
import sagemaker, boto3, os
sess = sagemaker.Session()
bucket = sess.default_bucket()
prefix = "airline-satisfaction"

s3 = boto3.client("s3")

# upload model.pkl
local_model_path = os.path.abspath("../model/model.pkl")
s3_key = f"{prefix}/model/model.pkl"
s3.upload_file(local_model_path, bucket, s3_key)

print("Uploaded:", f"s3://{bucket}/{s3_key}")

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
Uploaded: s3://amazon-sagemaker-913524921725-us-east-1-cp292gcmooo3l3/airline-satisfaction/model/model.pkl


In [12]:
import kagglehub, os, pandas as pd

path = kagglehub.dataset_download("teejmahal20/airline-passenger-satisfaction")
files = os.listdir(path)

if "train.csv" in files and "test.csv" in files:
    train_df = pd.read_csv(os.path.join(path, "train.csv"))
    test_df  = pd.read_csv(os.path.join(path, "test.csv"))
    df = pd.concat([train_df, test_df], ignore_index=True)
else:
    csvs = [f for f in files if f.endswith(".csv")]
    df = pd.read_csv(os.path.join(path, csvs[0]))

# Minimal cleaning for training job input
df = df.drop(columns=["Unnamed: 0", "id"], errors="ignore")
df.to_csv("train_full.csv", index=False)

print("Saved:", os.path.abspath("train_full.csv"), "rows:", df.shape[0])

Saved: /mnt/custom-file-systems/s3/shared/train_full.csv rows: 129880


In [13]:
import sagemaker, boto3

sess = sagemaker.Session()
bucket = sess.default_bucket()
prefix = "airline-satisfaction/data"

s3_uri = sess.upload_data("train_full.csv", bucket=bucket, key_prefix=prefix)
print("Uploaded to:", s3_uri)

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
Uploaded to: s3://amazon-sagemaker-913524921725-us-east-1-cp292gcmooo3l3/airline-satisfaction/data/train_full.csv


In [14]:
import os

os.makedirs("src", exist_ok=True)

train_script = r'''
import argparse
import os
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data-dir", type=str, default="/opt/ml/input/data/training")
    parser.add_argument("--model-dir", type=str, default=os.environ.get("SM_MODEL_DIR", "."))
    args = parser.parse_args()

    files = [f for f in os.listdir(args.data_dir) if f.endswith(".csv")]
    if not files:
        raise FileNotFoundError(f"No CSV found in {args.data_dir}. Found: {os.listdir(args.data_dir)}")

    csv_path = os.path.join(args.data_dir, files[0])
    df = pd.read_csv(csv_path)

    df = df.drop(columns=["Unnamed: 0", "id"], errors="ignore")
    df["satisfaction"] = df["satisfaction"].map({"satisfied": 1, "neutral or dissatisfied": 0})

    if df["satisfaction"].isna().any():
        raise ValueError("Target mapping produced NaNs. Check satisfaction values.")

    X = df.drop(columns=["satisfaction"])
    y = df["satisfaction"]

    cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
    num_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

    preprocess = ColumnTransformer(
        transformers=[
            ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                              ("scaler", StandardScaler())]), num_cols),
            ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                              ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_cols)
        ]
    )

    clf = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", LogisticRegression(max_iter=300))
    ])

    X_train, _, y_train, _ = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
    clf.fit(X_train, y_train)

    os.makedirs(args.model_dir, exist_ok=True)
    out_path = os.path.join(args.model_dir, "model.pkl")
    joblib.dump(clf, out_path)
    print("Saved model to:", out_path)

if __name__ == "__main__":
    main()
'''

with open("src/train_sagemaker.py", "w") as f:
    f.write(train_script)

print("Created:", os.path.abspath("src/train_sagemaker.py"))
print("src files:", os.listdir("src"))

Created: /mnt/custom-file-systems/s3/shared/src/train_sagemaker.py
src files: ['train_sagemaker.py']


In [15]:
import sagemaker
from sagemaker.sklearn.estimator import SKLearn

role = sagemaker.get_execution_role()

estimator = SKLearn(
    entry_point="train_sagemaker.py",
    source_dir="src",
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    framework_version="1.2-1",
    py_version="py3",
)

estimator.fit({"training": s3_uri})

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.Environment
2026-02-27 20:11:17 Starting - Starting the training job...
2026-02-27 20:11:33 Starting - Preparing the instances for training...
2026-02-27 20:11:59 Downloading - Downloading input data...
2026-02-27 20:12:24 Downloading - Downloading the training image...../miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_res

In [16]:
print("Model artifact (S3):", estimator.model_data)

Model artifact (S3): s3://amazon-sagemaker-913524921725-us-east-1-cp292gcmooo3l3/shared/sagemaker-scikit-learn-2026-02-27-20-11-14-686/output/model.tar.gz


In [17]:
import os, tarfile
import sagemaker

sess = sagemaker.Session()

local_tar = "model.tar.gz"
sess.download_data(path=".", bucket=estimator.model_data.split("/")[2],
                   key_prefix="/".join(estimator.model_data.split("/")[3:]))

# The download_data call saves into a folder sometimes; easiest approach:
print("Downloaded files:", os.listdir(".")) 

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
Downloaded files: ['.ipynb_checkpoints', '.libs.json', '.temp_sagemaker_unified_studio_debugging_info', 'airline-satisfaction.AWS.ipynb', 'model', 'sagemaker-scikit-learn-2026-02-27-05-35-50-152', 'sagemaker-scikit-learn-2026-02-27-12-46-43-766', 'sagemaker-scikit-learn-2026-02-27-12-57-42-635', 'sagemaker-scikit-learn-2026-02-27-13-28-57-079', 'sagemaker-scikit-learn-2026-02-27-20-11-14-686', 'sm_job', 'src', 'train_full.csv', 'model.tar.gz']


In [18]:
import tarfile, os
os.makedirs("model", exist_ok=True)

with tarfile.open("model.tar.gz", "r:gz") as tar:
    tar.extractall(path="model")

print("Extracted model folder contents:", os.listdir("model"))

Extracted model folder contents: ['model.pkl']


In [21]:
!pip uninstall -y sagemaker
!pip install sagemaker==2.224.0

Found existing installation: sagemaker 2.254.1
Uninstalling sagemaker-2.254.1:
  Successfully uninstalled sagemaker-2.254.1
  Using cached sagemaker-2.224.0-py3-none-any.whl.metadata (15 kB)
  Using cached attrs-23.2.0-py3-none-any.whl.metadata (9.5 kB)
  Using cached cloudpickle-2.2.1-py3-none-any.whl.metadata (6.9 kB)
  Using cached protobuf-4.25.8-cp37-abi3-manylinux2014_x86_64.whl.metadata (541 bytes)
Using cached sagemaker-2.224.0-py3-none-any.whl (1.5 MB)
Using cached cloudpickle-2.2.1-py3-none-any.whl (25 kB)
Using cached attrs-23.2.0-py3-none-any.whl (60 kB)
Using cached protobuf-4.25.8-cp37-abi3-manylinux2014_x86_64.whl (294 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.28.3
    Uninstalling protobuf-5.28.3:
      Successfully uninstalled protobuf-5.28.3
  Attempting uninstall: cloudpickle
    Found existing installation: cloudpickle 3.1.2
    Uninstalling cloudpickle-3.1.2:
      Successfully uninstalled cloudpickle-3.1.2
  Attempting uninst

In [22]:
import sagemaker
print("SageMaker version:", sagemaker.__version__)

SageMaker version: 2.254.1


In [23]:
import sagemaker

sess = sagemaker.Session()
bucket = sess.default_bucket()

print("Bucket:", bucket)

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
Bucket: amazon-sagemaker-913524921725-us-east-1-cp292gcmooo3l3


In [25]:
import sagemaker

sess = sagemaker.Session()
bucket = sess.default_bucket()
prefix = "airline-satisfaction"

# upload training file
sess.upload_data(
    path + "/train.csv",
    bucket=bucket,
    key_prefix=f"{prefix}/train"
)

train_s3_uri = f"s3://{bucket}/{prefix}/train/"
print("Train S3 URI:", train_s3_uri)

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
Train S3 URI: s3://amazon-sagemaker-913524921725-us-east-1-cp292gcmooo3l3/airline-satisfaction/train/


In [26]:
import os

os.makedirs("sm_job", exist_ok=True)

train_script = r'''
import os
import argparse
import joblib
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--train", type=str, default=os.environ.get("SM_CHANNEL_TRAIN", "/opt/ml/input/data/train"))
    parser.add_argument("--model-dir", type=str, default=os.environ.get("SM_MODEL_DIR", "/opt/ml/model"))
    parser.add_argument("--train-file", type=str, default="train.csv")  # ✅ airline train file
    parser.add_argument("--target-col", type=str, default="satisfaction")  # ✅ airline target
    parser.add_argument("--test-size", type=float, default=0.2)
    parser.add_argument("--random-state", type=int, default=42)
    return parser.parse_args()

def main():
    args = parse_args()

    data_path = os.path.join(args.train, args.train_file)
    if not os.path.exists(data_path):
        raise FileNotFoundError(f"Training file not found: {data_path}")

    df = pd.read_csv(data_path)

    # Drop common index-like columns if present
    for c in ["Unnamed: 0", "id", "ID"]:
        if c in df.columns:
            df = df.drop(columns=[c])

    if args.target_col not in df.columns:
        raise ValueError(f"Target column '{args.target_col}' not found. Columns: {list(df.columns)}")

    # Target mapping: "satisfied" vs others
    y_raw = df[args.target_col].astype(str).str.strip().str.lower()
    y = (y_raw == "satisfied").astype(int)  # 1=satisfied, 0=otherwise

    X = df.drop(columns=[args.target_col])

    # Identify column types
    numeric_cols = X.select_dtypes(include=["number", "int64", "float64"]).columns.tolist()
    categorical_cols = [c for c in X.columns if c not in numeric_cols]

    # Preprocessing
    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_cols),
            ("cat", categorical_transformer, categorical_cols),
        ]
    )

    # Model
    clf = LogisticRegression(max_iter=1000)

    model = Pipeline(steps=[
        ("preprocess", preprocessor),
        ("clf", clf)
    ])

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=args.test_size,
        random_state=args.random_state,
        stratify=y
    )

    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    acc = float(accuracy_score(y_test, preds))
    f1 = float(f1_score(y_test, preds))

    print(f"[Airline] Accuracy: {acc:.4f}")
    print(f"[Airline] F1-score:  {f1:.4f}")
    print("[Airline] Classification report:")
    print(classification_report(y_test, preds))

    # Save to /opt/ml/model for endpoint hosting
    os.makedirs(args.model_dir, exist_ok=True)
    joblib.dump(model, os.path.join(args.model_dir, "model.joblib"))
    print("Saved model to SM_MODEL_DIR")

if __name__ == "__main__":
    main()
'''

with open("sm_job/train_sagemaker.py", "w") as f:
    f.write(train_script)

print("Created sm_job/train_sagemaker.py")

Created sm_job/train_sagemaker.py


In [27]:
from sagemaker.sklearn.estimator import SKLearn
import sagemaker

role = sagemaker.get_execution_role()

estimator = SKLearn(
    entry_point="train_sagemaker.py",
    source_dir="sm_job",
    role=role,
    instance_type="ml.m5.large",
    framework_version="1.2-1",
    py_version="py3",
)

estimator.fit({"train": train_s3_uri})

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.Environment
2026-02-27 20:24:43 Starting - Starting the training job...
2026-02-27 20:24:59 Starting - Preparing the instances for training...
2026-02-27 20:25:23 Downloading - Downloading input data...
2026-02-27 20:25:53 Downloading - Downloading the training image......
2026-02-27 20:27:05 Training - Training image download completed. Training in progress.
2026-02-27 20:27:05 Uploading - Uploading generated training model/miniconda3/lib/python3.9/site-packages/sage

In [28]:
import os

os.makedirs("sm_job", exist_ok=True)

inference_script = r'''
import os
import json
import joblib
import pandas as pd

# Load model from /opt/ml/model
def model_fn(model_dir):
    model_path = os.path.join(model_dir, "model.joblib")
    model = joblib.load(model_path)
    return model

def input_fn(request_body, request_content_type):
    """
    Supported:
    - application/json:
        {"data": {"Gender":"Male", "Age":35, ...}}
        OR
        {"data": [{"Gender":"Male", "Age":35, ...}, {...}]}

    - text/csv:
        first line must be header:
        Gender,Age,Class,Type of Travel,...
        Male,35,Business,Business travel,...
    """
    if request_content_type == "application/json":
        payload = json.loads(request_body)

        if "data" not in payload:
            raise ValueError("JSON must include top-level key 'data'")

        data = payload["data"]

        # dict -> single row
        if isinstance(data, dict):
            return pd.DataFrame([data])

        # list[dict] -> multiple rows
        if isinstance(data, list):
            return pd.DataFrame(data)

        raise ValueError("'data' must be a dict or a list of dicts")

    if request_content_type == "text/csv":
        # Expect CSV WITH HEADER so we keep column names
        from io import StringIO
        return pd.read_csv(StringIO(request_body))

    raise ValueError(f"Unsupported content type: {request_content_type}")

def predict_fn(input_data, model):
    """
    Returns:
    - pred_class: 0/1
    - pred_proba: probability of class 1 (Satisfied)
    """
    pred_class = model.predict(input_data)

    # Pipeline ends with LogisticRegression -> supports predict_proba
    if hasattr(model, "predict_proba"):
        pred_proba = model.predict_proba(input_data)[:, 1]
    else:
        pred_proba = None

    return {"pred_class": pred_class.tolist(),
            "pred_proba": None if pred_proba is None else pred_proba.tolist()}

def output_fn(prediction, response_content_type):
    if response_content_type == "application/json":
        return json.dumps(prediction), "application/json"

    # default to json anyway
    return json.dumps(prediction), "application/json"
'''

with open("sm_job/inference.py", "w") as f:
    f.write(inference_script)

print("Created sm_job/inference.py")

Created sm_job/inference.py


In [30]:
# Deploy End Point

import sagemaker
import time
from sagemaker.sklearn.model import SKLearnModel
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer

role = sagemaker.get_execution_role()

model = SKLearnModel(
    model_data=estimator.model_data,   # from the completed training job
    role=role,
    entry_point="inference.py",
    source_dir="sm_job",
    framework_version="1.2-1",
    py_version="py3",
)

endpoint_name = f"airline-satisfaction-{int(time.time())}"

predictor = model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name=endpoint_name
)

# For airline, JSON is best (keeps feature names)
predictor.serializer = JSONSerializer()
predictor.deserializer = JSONDeserializer()

print("Endpoint:", endpoint_name)

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
------!Endpoint: airline-satisfaction-1772224179


In [31]:
sample = {
    "data": {
        "Gender": "Male",
        "Customer Type": "Loyal Customer",
        "Age": 35,
        "Type of Travel": "Business travel",
        "Class": "Business",
        "Flight Distance": 800,
        "Inflight wifi service": 4,
        "Departure/Arrival time convenient": 3,
        "Ease of Online booking": 4,
        "Gate location": 3,
        "Food and drink": 4,
        "Online boarding": 4,
        "Seat comfort": 4,
        "Inflight entertainment": 4,
        "On-board service": 4,
        "Leg room service": 4,
        "Baggage handling": 4,
        "Checkin service": 4,
        "Inflight service": 4,
        "Cleanliness": 4,
        "Departure Delay in Minutes": 5,
        "Arrival Delay in Minutes": 0
    }
}

result = predictor.predict(sample)
print(result)

{'pred_class': [1], 'pred_proba': [0.9501153407053274]}


In [33]:
# Inference

import pandas as pd
import numpy as np
import kagglehub, os

# get dataset path
path = kagglehub.dataset_download("teejmahal20/airline-passenger-satisfaction")

# load file
df = pd.read_csv(path + "/train.csv")
print("Loaded:", df.shape)

# drop index columns if present
for c in ["Unnamed: 0", "id", "ID"]:
    if c in df.columns:
        df = df.drop(columns=[c])

target_col = "satisfaction"

# prepare one row
X_row = df.drop(columns=[target_col]).iloc[0].to_dict()

# replace NaN for JSON safety
for k, v in list(X_row.items()):
    if isinstance(v, float) and np.isnan(v):
        X_row[k] = None

payload = {"data": X_row}

result = predictor.predict(payload)
print("Prediction result:", result)

actual = str(df[target_col].iloc[0])
print("Actual satisfaction:", actual)

Loaded: (103904, 25)
Prediction result: {'pred_class': [0], 'pred_proba': [0.20618470371188877]}
Actual satisfaction: neutral or dissatisfied


In [34]:
predictor.delete_endpoint()
print("Endpoint deleted")

Endpoint deleted
